# Verify Module 5: ASST-GCN, and the full LandmarkEncoder end-to-end

Two things: (1) `ASSTGCN` in isolation, checking the semantic/spatio-
temporal adjacency mechanics (Eq. 4-6 in Sheng et al. 2022 -- see
`fusion_avsr/models/landmark/asst_gcn.py`'s module docstring for the
exact equations as transcribed from the paper) actually behave as
specified; (2) the full `LandmarkEncoder` (Modules 1-5 wired together)
run end-to-end on a real clip from each of the three sources.

**What "looks right" means:**
- Each subgraph's spatio-temporal adjacency `A^st_q` is row-stochastic
  (softmax output: every row sums to 1).
- Each subgraph's semantic adjacency `A^se_q` starts at the paper's
  specified constant (1e-6) before any training.
- `LandmarkEncoder` produces `(1, T, 512)` on a real clip from every
  source, with `T` matching the clip's frame count, no `NaN`/`Inf`.
- A backward pass reaches every parameter across all 5 modules.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # adjust if running from somewhere else
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# TODO: fill in correct paths (same convention as notebook 01)
PROJECT_NAME = "your_project_name"
DATASET_ROOT = Path(f"/scratch/{PROJECT_NAME}/datasets")
LRS3_ROOT = DATASET_ROOT / "lrs3"

grid_all_path = "kaggle_lipnet/datasets/jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet/versions/1/data"
GRID_ROOT = DATASET_ROOT / grid_all_path
GRID_LANDMARKS_ROOT = DATASET_ROOT / "grid_landmarks"

LRS3_TRAINVAL_LANDMARKS_ROOT = LRS3_ROOT / "landmarks" / "LRS3_landmarks" / "trainval"
LRS3_TRAINVAL_VIDEO_ROOT = LRS3_ROOT / "ainncy" / "trainval"

LRS3_TEST_VIDEO_ROOT = LRS3_ROOT / "test"
LRS3_TEST_LANDMARKS_ROOT = LRS3_ROOT / "landmarks" / "LRS3_landmarks" / "test"

AUDIO_OUTPUT_DIR = DATASET_ROOT / "extracted_audio"
LIMIT = 5  # a handful of clips per source, not the whole dataset

SEED = 42

In [ ]:
import pickle

import numpy as np
import pandas as pd
import torch
from torchcodec.decoders import VideoDecoder

from fusion_avsr.data.manifest_builder import build_grid_manifest, build_lrs3_manifest, load_or_build_manifest
from fusion_avsr.data.paths import MANIFEST_DIR

# These are cached under MANIFEST_DIR: built once (here or by notebook 01,
# whichever runs first), loaded from the cached CSV every time after.
grid_manifest = load_or_build_manifest(
    MANIFEST_DIR / "grid_manifest.csv", build_grid_manifest,
    grid_root=GRID_ROOT, landmarks_root=GRID_LANDMARKS_ROOT,
    audio_output_dir=AUDIO_OUTPUT_DIR, limit=LIMIT,
)
lrs3_trainval_manifest = load_or_build_manifest(
    MANIFEST_DIR / "lrs3_trainval_manifest.csv", build_lrs3_manifest,
    video_root=LRS3_TRAINVAL_VIDEO_ROOT, audio_output_dir=AUDIO_OUTPUT_DIR,
    landmarks_root=LRS3_TRAINVAL_LANDMARKS_ROOT, source="lrs3_trainval", limit=LIMIT,
)
lrs3_test_manifest = load_or_build_manifest(
    MANIFEST_DIR / "lrs3_test_manifest.csv", build_lrs3_manifest,
    video_root=LRS3_TEST_VIDEO_ROOT, audio_output_dir=AUDIO_OUTPUT_DIR,
    landmarks_root=LRS3_TEST_LANDMARKS_ROOT, source="lrs3_test", limit=LIMIT,
)

MANIFESTS = {
    "grid": grid_manifest,
    "lrs3_trainval": lrs3_trainval_manifest,
    "lrs3_test": lrs3_test_manifest,
}
for name, m in MANIFESTS.items():
    print(name, m.shape)

In [ ]:
from fusion_avsr.utils.video import decode_all_frames


def load_one_clip(manifest, seed=SEED):
    """Pick one random row and decode its frames + landmarks."""
    row = manifest.sample(n=1, random_state=seed).iloc[0]
    decoder = VideoDecoder(row["video_path"], dimension_order="NHWC")
    frames = decode_all_frames(decoder)
    with open(row["landmark_path"], "rb") as f:
        landmarks = pickle.load(f)
    n = min(len(frames), len(landmarks))
    return row["sample_id"], frames[:n], landmarks[:n]

In [ ]:
from fusion_avsr.models.landmark.lrlp import align_to_nose_tip, extract_lrlp_sequence

def get_module1_outputs(manifest, seed=SEED):
    sample_id, frames, landmarks = load_one_clip(manifest, seed=seed)
    patches, raw_coords, valid_mask = extract_lrlp_sequence(frames, landmarks)
    aligned_coords = align_to_nose_tip(raw_coords, landmarks)
    return sample_id, patches, raw_coords, aligned_coords, valid_mask

## Part 1: ASST-GCN mechanics, synthetic input

In [ ]:
import torch

from fusion_avsr.models.landmark.asst_gcn import ASST_GCN_CHANNELS, SEMANTIC_GRAPH_INIT_VALUE, ASSTGCNLayer
from fusion_avsr.models.landmark.lrlp import NUM_LRLPS

layer = ASSTGCNLayer(channels=ASST_GCN_CHANNELS, num_subgraphs=8, num_nodes=NUM_LRLPS)

# A^se starts at the paper's specified constant.
assert torch.allclose(layer.semantic_adjacency, torch.full_like(layer.semantic_adjacency, SEMANTIC_GRAPH_INIT_VALUE))
print("OK: semantic adjacency initialized to", SEMANTIC_GRAPH_INIT_VALUE)

# A^st is row-stochastic (a softmax output).
x = torch.randn(1, 3, NUM_LRLPS, ASST_GCN_CHANNELS)
theta = layer.theta_proj[0](x)
phi = layer.phi_proj[0](x)
similarity = theta @ phi.transpose(-1, -2)
a_st = torch.softmax(similarity, dim=-1)
row_sums = a_st.sum(dim=-1)
print("A^st row sums (should all be ~1.0):", row_sums.flatten()[:5])
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5)

## Part 2: full LandmarkEncoder on one real clip from each source

In [ ]:
from fusion_avsr.models.landmark.encoder import LandmarkEncoder

encoder = LandmarkEncoder(output_dim=None)
print(f"total parameters: {sum(p.numel() for p in encoder.parameters()):,}")

In [ ]:
for source_name, manifest in MANIFESTS.items():
    sample_id, patches, raw_coords, aligned_coords, valid_mask = get_module1_outputs(manifest)
    patches_tensor = torch.from_numpy(patches).float().unsqueeze(0)
    coords_tensor = torch.from_numpy(aligned_coords).float().permute(0, 2, 1).unsqueeze(0)

    output = encoder(patches_tensor, coords_tensor)
    print(f"{source_name} ({sample_id}): T={patches_tensor.shape[2]} -> output={tuple(output.shape)}")

    assert output.shape == (1, patches_tensor.shape[2], 512)
    assert torch.isfinite(output).all()

## Backward pass reaches every parameter, across all 5 modules

In [ ]:
sample_id, patches, raw_coords, aligned_coords, valid_mask = get_module1_outputs(MANIFESTS["grid"])
patches_tensor = torch.from_numpy(patches).float().unsqueeze(0)
coords_tensor = torch.from_numpy(aligned_coords).float().permute(0, 2, 1).unsqueeze(0)

output = encoder(patches_tensor, coords_tensor)
output.sum().backward()

missing_grad = [name for name, p in encoder.named_parameters() if p.grad is None]
print(f"{len(missing_grad)} parameters with no gradient (should be 0):")
print(missing_grad)
assert not missing_grad

## Checklist

- [ ] `A^se` starts at 1e-6, `A^st` rows sum to 1 (asserted above).
- [ ] `LandmarkEncoder` output is `(1, T, 512)` with `T` matching each clip's frame count, for GRID, LRS3-trainval, AND LRS3-test.
- [ ] No `NaN`/`Inf` in any output.
- [ ] No parameters missing a gradient after the backward pass (confirms LMFE, LCFE, the semantic embedding, and all 6 ASST-GCN layers are all actually wired into the computation graph).